### Миграция данных с PostgreSQL в Clickhouse для ускорение дальнейших аналитических запросов (OLAP)
- Postgres (ETL) -> Jupyter (python script) -> Clickhouse (DWH)

Этот python скрипт одноразовый, автоматизированный ETL пайплайн не требуется, так как данные статичны.\
Подключимся к базе данных postgres через библиотеку sqlalchemy, а к Clickhouse через clickhouse_connect.

In [1]:
# Импорт необходимых библиотек
import pandas as pd
from sqlalchemy import create_engine
import os
from clickhouse_driver import Client
import clickhouse_connect

In [32]:
# Данные для подключения к PostgreSQL
pg_user = os.getenv('POSTGRES_USER')
pg_password = os.getenv('POSTGRES_PASSWORD')
pg_host = 'localhost'
pg_port = '5432'
pg_database = 'postgres'

# Данные для подключения к ClickHouse
ch_host = 'localhost'
ch_port = '8123'
ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PASSWORD')
ch_database = 'dwh'

In [44]:
# Создание подключения к PostgreSQL и ClickHouse
pg_engine = create_engine(
    f"postgresql+psycopg2://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_database}", echo=False
)
ch_connect = clickhouse_connect.get_client(
    host=ch_host,
    port=ch_port,
    username=ch_user,
    password=ch_password,
    database=ch_database
)

In [50]:
# Чтение данных из PostgreSQL
df = pd.read_sql("""
        SELECT o.order_id, oi.row_id, o.order_date, 
            cu.customer_name, s.segment, c.city,
            st.state, r.region, p.product_name, pc.category,
            sc.sub_category, sm.ship_mode, ss.ship_status,
            oi.profit, oi.quantity, oi.discount, oi.sales, oi.sales_forecast
    FROM stage.orders o
    JOIN stage.order_items oi ON o.order_id = oi.order_id
    JOIN stage.products p ON oi.product_id = p.product_id
    JOIN stage.sub_categories sc ON p.sub_category_id = sc.sub_category_id
    JOIN stage.categories pc ON sc.category_id = pc.category_id
    JOIN stage.cities c ON o.city_id = c.city_id
    JOIN stage.states st ON c.state_id = st.state_id
    JOIN stage.regions r ON st.region_id = r.region_id
    JOIN stage.customers cu ON o.customer_id = cu.customer_id
    JOIN stage.segments s ON cu.segment_id = s.segment_id
    JOIN stage.ship_modes sm ON o.ship_mode_id = sm.ship_mode_id
    JOIN stage.ship_statuses ss ON o.ship_status_id = ss.ship_status_id
""", pg_engine)
pg_engine.dispose()

In [52]:
# Загрузка данных в ClickHouse
ch_connect.insert_df('dwh.sales_fact', df)
print("Данные успешно загружены в ClickHouse")

# Проверка данных в ClickHouse
result = ch_connect.query("SELECT COUNT(*) FROM dwh.sales_fact")
print(f"Количество записей в ClickHouse: {result.result_rows[0][0]}")

Данные успешно загружены в ClickHouse
Количество записей в ClickHouse: 9994
